### Detect spots with TrackPy and save CSVs ready for MakeCropArrays.ipynb.

In [ ]:
import croparray as ca
from pathlib import Path

In [ ]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

DATA_DIR   = Path(r"Z:\SharedInternal\RiboJamProject\20260508_deltaZNF_LightInducedJams\WT_U2OS")
VID_FILES  = sorted(DATA_DIR.glob("*.tif"))

DETECT_CH   = 2     # Channel index (0-based) to use for spot detection
DIAMETER    = 7    # Estimated spot diameter in pixels (must be odd)
SEPARATION  = 5    # How close two spots can be together and still called as separate

FRAME_INDEX = None  # Frame for preview/threshold review; None = middle frame
MAX_SPOTS   = 200   # Safety cap: auto-threshold will never exceed this many spots/frame

TRACK          = True  # Link spots into tracks?
SEARCH_RANGE   = 5     # Max displacement (pixels) between frames
MEMORY         = 2     # Frames a spot can vanish and still be linked
MIN_TRACK_LEN  = 3     # Drop tracks shorter than this (frames)

SPOTS_SUFFIX = "_spots"   # Output: <video_stem>_spots.csv  (all detections)
TRACKS_SUFFIX = "_tracks" # Output: <video_stem>_tracks.csv (linked tracks, if TRACK=True)
OUT_DIR      = DATA_DIR   # Where to save CSVs

In [ ]:
# Preview a single video to check DIAMETER. Axis-labeling GUI will open first.
ca.build.preview_detection(VID_FILES[0], detect_ch=DETECT_CH,
                            diameter=DIAMETER, frame_index=FRAME_INDEX,
                            max_spots=MAX_SPOTS, separation = SEPARATION)

In [ ]:
# (Optional) Draw polygon ROIs to exclude spots outside regions of interest.
# Opens napari showing max(t,z) projection for each video — scroll through frames,
# draw one polygon per frame using the Polygon tool (P), then close the window.
# Saves <video_stem>__roi.json sidecars; make_csvs applies them automatically.
rois = ca.build.review_rois(VID_FILES, detect_ch=DETECT_CH)

In [ ]:
# Review and adjust per-video thresholds.
# Use the dropdown to switch videos, slider to adjust minmass.
# Changes are saved automatically — just run the next cell when done.
thresholds = ca.build.review_thresholds(VID_FILES, detect_ch=DETECT_CH,
                                         diameter=DIAMETER, frame_index=FRAME_INDEX,
                                         max_spots=MAX_SPOTS, separation = SEPARATION)

In [ ]:
# Batch detect + track all videos using the reviewed thresholds, then save CSVs.
# Always saves: <stem>_spots.csv  (all detections)
# If TRACK=True: <stem>_tracks.csv (linked + filtered tracks)
ca.build.make_csvs(VID_FILES, detect_ch=DETECT_CH, diameter=DIAMETER,
                   minmass=thresholds, track=TRACK, search_range=SEARCH_RANGE,
                   memory=MEMORY, min_track_len=MIN_TRACK_LEN,
                   spots_suffix=SPOTS_SUFFIX, tracks_suffix=TRACKS_SUFFIX,
                   out_dir=OUT_DIR, separation = SEPARATION)

In [ ]:
df18 = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell24_spots.csv')
df19 = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell25_spots.csv')

df19['frame'] = df19['frame'] + 300

df_all = pd.concat([df18, df19], ignore_index=True)

g = sns.relplot(data=df_all, x='frame', y='mass', kind='line', height=4, aspect=2)
g.set_axis_labels('Frame', 'Spot Intensity (mass)')
plt.title('Average Spot Intensity Over Time')

In [ ]:
df18 = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell18_spots.csv')
df19 = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell19_spots.csv')

df19['frame'] = df19['frame'] + 300

df_all = pd.concat([df18, df19], ignore_index=True)

g = sns.relplot(data=df_all, x='frame', y='mass', kind='line', height=4, aspect=2)
g.set_axis_labels('Frame', 'Spot Intensity (mass)')
plt.title('Average Spot Intensity Over Time')


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell41_spots.csv')

g = sns.relplot(data=df, x='frame', y='mass', kind='line', height=4, aspect=1.5)
g.set_axis_labels('Frame', 'Spot Intensity (mass)')
plt.title('Average Spot Intensity Over Time')


In [ ]:
df18_tracks = pd.read_csv(r'Z:/SharedInternal/RiboJamProject/20260508_deltaZNF_LightInducedJams/WT_U2OS/Cell35_tracks.csv')

track_lengths = df18_tracks.groupby('particle')['frame'].count()
long_tracks = track_lengths[track_lengths > 100].index

df_long = df18_tracks[df18_tracks['particle'].isin(long_tracks)]

g = sns.relplot(data=df_long, x='frame', y='mass', kind='line', 
                hue='particle', height=4, aspect=2, legend=False, palette='tab10',col='particle',col_wrap=6)
g.set_axis_labels('Frame', 'Spot Intensity (mass)')
plt.title('Long Track Intensities (>100 frames)')
